### Base Model

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score,GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix,roc_auc_score,RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [17]:
df = pd.read_csv("dataset/processed/final_df.csv")

X = df.drop(["delivered_late", "order_delivered_customer_date", "order_estimated_delivery_date","order_id",               
"customer_id" ,                   
"order_status",                  
"order_purchase_timestamp"    ,  
"order_approved_at"           ,  
"order_delivered_carrier_date"  ,   
"customer_city"                ,  
"customer_state",
'delivery_time_days',            
'carrier_delay_days',           
'delivery_diff_estimated_days'            
            ], axis=1,errors='ignore')
y = df["delivered_late"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [18]:
# Logistic Regression
log_reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)

In [50]:
# Performans Sonuçları
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cf=confusion_matrix(y_test, y_pred)



print("Base Logistic Regression Accuracy:", accuracy)
print("Base Logistic Regression F1 Score:", f1)
print("Base Logistic Regression Confusion Matrix", cf)

Base Logistic Regression Accuracy: 0.8124051401396337
Base Logistic Regression F1 Score: 0.3432518597236982
Base Logistic Regression Confusion Matrix [[15089  3113]
 [  595   969]]


 ### ADVANCED MODELS
 

### XGBoost

In [20]:
# XGBoost
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

xgb_pipeline = Pipeline(steps=[
    ("model", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.4,
        colsample_bytree=0.6,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42
    ))
])

xgb_pipeline.fit(X_train, y_train)
y_pred = xgb_pipeline.predict(X_test)
y_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

In [22]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cf=confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC Score:", roc_auc)
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Confusion Matrix", cf)

ROC-AUC Score: 0.7635420638973093
Accuracy: 0.8097743600121421
F1 Score: 0.3405822518414591
Confusion Matrix [[15035  3167]
 [  593   971]]


In [26]:
#Hyperparameter Tuning
pipeline = Pipeline(steps=[
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        use_label_encoder=False
    ))
])

param_grid = {
    "model__n_estimators": [100,200, 300],
    "model__max_depth": [2,4, 6,8],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.6,0.8],
    "model__colsample_bytree": [0.6,0.8,1.0]
}
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=2
)
grid_search.fit(X_train, y_train)
print("Best ROC-AUC Score:", grid_search.best_score_)
print("Best Parameters:")
print(grid_search.best_params_)


Fitting 3 folds for each of 144 candidates, totalling 432 fits
Best ROC-AUC Score: 0.771962651705131
Best Parameters:
{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 200, 'model__subsample': 0.8}


C:\ProgramData\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [20:02:10] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [27]:
#Hyperparameter Tuning değerlerini kullanıcaz
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

xgb_pipeline = Pipeline(steps=[
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1.0,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42
    ))
])

xgb_pipeline.fit(X_train, y_train)
y_pred = xgb_pipeline.predict(X_test)
y_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

In [28]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cf=confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC Score:", roc_auc)
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Confusion Matrix", cf)

ROC-AUC Score: 0.7655603526888224
Accuracy: 0.801578468076495
F1 Score: 0.3350288233299423
Confusion Matrix [[14856  3346]
 [  576   988]]


### Random Forest

In [33]:
rf_pipeline = Pipeline(steps=[
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",  
        n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)
y_proba = rf_pipeline.predict_proba(X_test)[:, 1]

In [34]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cf=confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC Score:", roc_auc)
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Confusion Matrix", cf)

ROC-AUC Score: 0.6971712693667063
Accuracy: 0.8863705352625721
F1 Score: 0.2287087912087912
Confusion Matrix [[17187  1015]
 [ 1231   333]]


In [35]:
# Hyperparameter Tuning
rf_pipeline = Pipeline(steps=[
    ("rf", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 20],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features": ["sqrt", "log2"]
}
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

best_rf_model = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV ROC-AUC:")
print(grid_search.best_score_)


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters:
{'rf__max_depth': 5, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 5, 'rf__n_estimators': 100}
Best CV ROC-AUC:
0.770919172191942


In [38]:
#Hyperparameter Tuning değerlerini kullanıcaz
rf_pipeline = Pipeline(steps=[
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",  
        n_jobs=-1,
        max_depth= 5,
        max_features= 'sqrt', 
        min_samples_leaf= 2, 
        min_samples_split= 5,
       
    ))
])
rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)
y_proba = rf_pipeline.predict_proba(X_test)[:, 1]

In [39]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cf=confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC Score:", roc_auc)
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Confusion Matrix", cf)

ROC-AUC Score: 0.7635616649023421
Accuracy: 0.8179196600222605
F1 Score: 0.34623069936421436
Confusion Matrix [[15214  2988]
 [  611   953]]


### LİGHTGBM

In [42]:
lgbm_pipeline = Pipeline(steps=[
    ("lgbm", LGBMClassifier(
        objective="binary",
        random_state=42,
        class_weight="balanced",   
        n_estimators=300,
        learning_rate=0.05,
        n_jobs=-1
    ))
])
lgbm_pipeline.fit(X_train, y_train)


[LightGBM] [Info] Number of positive: 6255, number of negative: 72808
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002527 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 771
[LightGBM] [Info] Number of data points in the train set: 79063, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Pipeline(steps=[('lgbm',
                 LGBMClassifier(class_weight='balanced', learning_rate=0.05,
                                n_estimators=300, n_jobs=-1, objective='binary',
                                random_state=42))])

In [43]:
y_pred = lgbm_pipeline.predict(X_test)
y_proba = lgbm_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8163007184053425
ROC-AUC: 0.7604134905778881
F1 Score: 0.3432808826189184
Confusion Matrix:
 [[15186  3016]
 [  615   949]]


In [44]:
#HyperparameterTuning
lgbm_pipeline = Pipeline(steps=[
    ("lgbm", LGBMClassifier(
        objective="binary",
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])
param_grid = {
    "lgbm__n_estimators": [200, 300, 500],
    "lgbm__learning_rate": [0.01, 0.05, 0.1],
    "lgbm__max_depth": [-1, 5, 10],
    "lgbm__num_leaves": [15, 31, 63],
    "lgbm__min_child_samples": [20, 50, 100],
    "lgbm__subsample": [0.7, 0.8, 1.0],
    "lgbm__colsample_bytree": [0.7, 0.8, 1.0]
}
grid_search_lgbm = GridSearchCV(
    estimator=lgbm_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=2
)
grid_search_lgbm.fit(X_train, y_train)

best_lgbm_model = grid_search_lgbm.best_estimator_

print("Best Parameters:")
print(grid_search_lgbm.best_params_)

print("Best CV ROC-AUC:")
print(grid_search_lgbm.best_score_)


Fitting 5 folds for each of 2187 candidates, totalling 10935 fits
[LightGBM] [Info] Number of positive: 6255, number of negative: 72808
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000477 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 771
[LightGBM] [Info] Number of data points in the train set: 79063, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Best Parameters:
{'lgbm__colsample_bytree': 1.0, 'lgbm__learning_rate': 0.01, 'lgbm__max_depth': 10, 'lgbm__min_child_samples': 100, 'lgbm__n_estimators': 500, 'lgbm__num_leaves': 15, 'lgbm__subsample': 0.7}
Best CV ROC-AUC:
0.7726177275779996


In [47]:
#HyperparameterTuning bestparameters'lar denendi
lgbm_pipeline = Pipeline(steps=[
    ("lgbm", LGBMClassifier(
        objective="binary",
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
        colsample_bytree= 1.0, 
        learning_rate= 0.01, 
        max_depth= 10, 
        min_child_samples= 100,
        n_estimators= 500,
        num_leaves= 15,
        subsample= 0.7
    ))
])
lgbm_pipeline.fit(X_train, y_train)
y_pred = lgbm_pipeline.predict(X_test)
y_proba = lgbm_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

[LightGBM] [Info] Number of positive: 6255, number of negative: 72808
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000289 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 771
[LightGBM] [Info] Number of data points in the train set: 79063, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Accuracy: 0.8124051401396337
ROC-AUC: 0.7630533736069587
F1 Score: 0.3432518597236982
Confusion Matrix:
 [[15089  3113]
 [  595   969]]


In [52]:
X.columns

Index(['price', 'freight_value', 'review_score', 'price_freight_ratio'], dtype='object')